# NB_patient_360

Builds `adb_healthy.gold.patient_360` (one row per patient) from the silver layer per the STTM Silver to Gold document. Driving table `silver.patients`, LEFT JOINed to hospitals, insurance providers and per-patient lab result aggregates. Full rebuild on every run.

Note: `silver.lab_results` contains exact duplicate rows for some `lab_result_id`s, so it is deduplicated (latest `update_timestamp` kept) before aggregation.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_healthy.gold;

In [ ]:
%sql
CREATE OR REPLACE TABLE adb_healthy.gold.patient_360 AS
WITH lab_results_dedup AS (
  -- silver.lab_results has duplicate lab_result_id rows; keep the latest version
  SELECT *
  FROM adb_healthy.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM lab_results_dedup
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                             AS patient_id,
  p.first_name                                                          AS first_name,
  p.last_name                                                           AS last_name,
  CAST(p.date_of_birth AS DATE)                                         AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                              AS gender,
  p.city                                                                AS city,
  p.state                                                               AS state,
  CAST(p.registered_hospital_id AS INT)                                 AS hospital_id,
  h.hospital_name                                                       AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                  AS insurance_provider_id,
  ip.provider_name                                                      AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                              AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                      AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                      AS critical_test_count,
  la.last_test_date                                                     AS last_test_date
FROM adb_healthy.silver.patients p
LEFT JOIN adb_healthy.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_healthy.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [ ]:
updated_df = spark.table("adb_healthy.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))